# Structs

* A *struct* assembles several values of assorted types together into a single value, so you can deal with them as a unit. 
* Given a struct, you can read and modify its individual components. 
* A struct can have methods associated with it that operate on its components.

## Named-Field Structs

### Declaring a Struct

In [2]:
// A rectangle of eight-bit grayscale pixels.
struct GrayscaleMap {
    pixels: Vec<u8>,
    size: (usize, usize)
}

### Creating a Struct Value

In [3]:
let width = 800;
let height = 600;

let image = GrayscaleMap {
    pixels: vec![0; width * height],
    size: (width, height)
};

In [4]:
fn create_image(size: (usize, usize)) -> GrayscaleMap {
    let pixels = vec![0; size.0 * size.1];
    GrayscaleMap{
        pixels,
        size
    } // struct expression
}

### Accessing Struct Fields

In [5]:
let image: GrayscaleMap = create_image((800, 600));

assert!(image.size == (800, 600));
assert!(image.pixels.len() == image.size.0 * image.size.1);

### Visibility of Struct

* Structs are private by default, visible only in the module where they’re declared. 
* You can make a struct visible outside its module by prefixing its definition with `pub`

In [ ]:
// A rectangle of eight-bit grayscale pixels.
mod graphics {
    pub struct GrayscaleBitmap {
        pub pixels: Vec<u8>,
        pub size: (usize, usize)
    }

    pub fn create_image(size: (usize, usize)) -> GrayscaleBitmap {
        let pixels = vec![0; size.0 * size.1];
        GrayscaleBitmap{
            pixels,
            size
        } // struct expression
    }

    pub BoundingBox(pub usize, pub usize);
}

In [16]:
use graphics::{GrayscaleBitmap, create_image};

let img: GrayscaleBitmap = create_image((800, 600));

assert_eq!(img.size, (800, 600));
assert_eq!(img.pixels.len(), 800 * 600);

## Tuple-Like Structs

* The second kind of struct type is called a tuple-like struct, because it resembles a tuple:

In [18]:
struct BoundingBox(usize, usize);

let image_bbox = BoundingBox(800, 600);

assert!(image_bbox.0 * image_bbox.1 == 480_000);

* Individual elements of a tuple-like struct may be public or not:

In [19]:
pub struct BoundingBox(pub usize, pub usize);

### Tuple-Like Struct as Newtypes

* Tuple-like structs allow you to create types that behave like tuples but have a name and can implement traits
* This allows you to create newtypes for stricter type checking
* This is a workaround for the *orphan rule* - [New Type Pattern](https://doc.rust-lang.org/book/ch20-02-advanced-traits.html#using-the-newtype-pattern-to-implement-external-traits)

In [29]:
#[derive(Debug, Clone, Copy, PartialEq, PartialOrd)]
struct Distance(f64);

impl Distance {
    fn value(&self) -> f64 {
        self.0
    }
}

#[derive(Debug, Clone, Copy, PartialEq, PartialOrd)]
struct Velocity(f64);

impl Velocity {
    fn value(&self) -> f64 {
        self.0
    }
}

#[derive(Debug, Clone, Copy, PartialEq, PartialOrd)]
struct Time(f64);

impl Time {
    fn value(&self) -> f64 {
        self.0
    }
}

fn dangerous_speed(distance: f64, time: f64) -> f64 {
    distance / time
}

// A newtype for stricter type checking
fn calculate_distance(velocity: Velocity, time: Time) -> Distance {
    Distance(velocity.0 * time.0)
}

let velocity = Velocity(10.0);

let velocity_value = velocity.value();


let time = Time(5.0);
let distance = calculate_distance(velocity, Time(5.0));

println!("Distance traveled: {:?} kilometers", distance);
println!("Velocity: {:?} kilometers", velocity);

assert!(velocity < Velocity(20.0));

Distance traveled: Distance(50.0) kilometers
Velocity: Velocity(10.0) kilometers


## Unit-Like Structs

* The third kind of struct is a little obscure: it declares a struct type with no elements at all:

In [ ]:
struct Onesuch;

let o: Onesuch = Onesuch;

* This allows to create types that can act as tags or markers

## Defining Methods - `impl`

In [2]:
pub struct Queue {
    older: Vec<char>,
    younger: Vec<char>
}

impl Queue {
    pub fn push(&mut self, c: char) {
        self.younger.push(c);
    }

    pub fn pop(&mut self) -> Option<char> {
        if self.older.is_empty() {
            if self.younger.is_empty() {
                return None;
            }

            use std::mem::swap;
            swap(&mut self.older, &mut self.younger);
            self.older.reverse();
        }

        self.older.pop()
    }

    pub fn is_empty(&self) -> bool {
        self.older.is_empty() && self.younger.is_empty()
    }
}

In [3]:
let mut q = Queue{ older: Vec::new(), younger: Vec::new() };

q.push('@');
q.push('$');
assert!(q.pop() == Some('@'));

q.push('*');
assert!(q.pop() == Some('$'));
assert!(q.pop() == Some('*'));
assert!(q.pop() == None);

assert!(q.is_empty());

In [4]:
impl Queue {
    pub fn split(self) -> (Vec<char>, Vec<char>) {
        (self.older, self.younger)
    }
}

In [5]:
let mut q = Queue { older: Vec::new(), younger: Vec::new() };
q.push('Q');
q.push('U');
assert!(q.pop() == Some('Q'));
q.push('X');

let (older, younger) = q.split();

assert_eq!(older, vec!['U']);
assert_eq!(younger, vec!['X']);

### `new` Method

* Constructor function - doesn't take `self` as an argument (**static function**)

In [6]:
impl Queue {
    pub fn new() -> Self {
        Queue { older: Vec::new(), younger: Vec::new() }
    }
}

In [7]:
let mut q = Queue::new();

q.push('a');
q.push('b');
q.push('c');

assert_eq!(q.pop(), Some('a'));
assert_eq!(q.pop(), Some('b'));
assert_eq!(q.pop(), Some('c'));
assert_eq!(q.pop(), None);

## Generic Structs

* Generics allow you to write flexible, reusable functions and types that can work with any data type. Instead of specifying concrete types, you use type parameters, which are placeholders for types that will be specified when the function or type is used.
* Generics use parameters passed in angle brackets to specify the types they work with. For example, `fn example<T>(param: T) {}` uses `T` as a generic type parameter.

In [9]:
pub struct Stack<T> {
    items: Vec<T>
}

impl<T> Stack<T> {
    pub fn new() -> Self {
        Stack{ items: Vec::new() }
    }

    pub fn push(&mut self, item: T) {
        self.items.push(item)
    }

    pub fn pop(&mut self) -> Option<T> {
        self.items.pop()
    }

    pub fn is_empty(&self) -> bool {
        self.items.is_empty()
    }
}

* For static method calls, you can supply the type parameter explicitly using the turbofish `::<>` notation:

In [26]:
let s = Stack::<char>::new();

* But in practice, Rust can figure it out for you:

In [12]:
let mut stack_1: Stack<&str> = Stack::new();
let mut stack_2: Stack<f64> = Stack::new();

stack_1.push("Hello");
stack_2.push(3.1415);

stack_1.push("World");
stack_2.push(2.71);

## Deriving Common Traits for Structs

* Structs are not copyable, comparable, printable, or hashable by default

In [17]:
struct Point {
    x: i32,
    y: i32
}

let p1 = Point { x: 3, y: 4 };

println!("p1: {:?}", p1);

// let p2 = p1;
// println!("p1: ({}, {}), p2: ({}, {})", p1.x, p1.y, p2.x, p2.y);


Error: `Point` doesn't implement `Debug`

* You can derive the `Copy` and `Clone` traits for a struct type, but only if all its fields are copyable or clonable
* You can derive the `PartialEq` and `Eq` traits for a struct type, but only if all its fields are comparable for equality
* You can derive the `Ord` and `PartialOrd` traits for a struct type, but only if all its fields are comparable for ordering
* You can derive `Debug` for a struct type, but only if all its fields are printable

In [18]:
#[derive(Copy, Clone, Debug, PartialEq)]
struct Point {
    x: f64,
    y: f64
}

In [20]:
let mut pt = Point{ x: 0.0, y: 1.0 };
let other_pt = pt;

println!("pt = {:?}", pt);
println!("other_pt = {:?}", other_pt);

assert!(pt == other_pt);

let another_pt = pt.clone();

pt.x = 42.42;
assert!(pt != other_pt);
assert!(pt != another_pt);

println!("pt = {:?}", pt);
println!("other_pt = {:?}", other_pt);
println!("another_pt = {:?}", another_pt);

pt = Point { x: 0.0, y: 1.0 }
other_pt = Point { x: 0.0, y: 1.0 }
pt = Point { x: 42.42, y: 1.0 }
other_pt = Point { x: 0.0, y: 1.0 }
another_pt = Point { x: 0.0, y: 1.0 }


In [37]:
#[derive(Debug, Clone)]
struct Container(Vec<i32>);

#[derive(Debug, Clone)]
struct Data {
    name: String,
    data: Container,
    timestamp: u64,
}

let data_instance = Data {
    name: String::from("Sample Data"),
    data: Container(vec![1, 2, 3, 4, 5]),
    timestamp: 1234567890,
};

println!("Data instance: {:?}", data_instance);

let another_snapshot = Data {
    timestamp: 7623547623,
    ..data_instance // Use the rest of the fields from data_instance
};

println!("Another snapshot: {:#?}", another_snapshot);

Data instance: Data { name: "Sample Data", data: Container([1, 2, 3, 4, 5]), timestamp: 1234567890 }
Another snapshot: Data {
    name: "Sample Data",
    data: Container(
        [
            1,
            2,
            3,
            4,
            5,
        ],
    ),
    timestamp: 7623547623,
}
